### 응용 미니프로젝트
- 파일 : n2-5_project.ipynb
- 역할 : 파이프라인 전체를 하나의 응용 프로그램으로 통합하고 최적화 전후를 비교합니다.
- 기능 :
    1. 프로젝트 설정  
    2. 입력 모듈: 프레임 선택  
    3. 백엔드별 입력 형식 만들기  
    4. CPU 백엔드  
    5. NPU 백엔드  
    6. 두 경로의 결과 비교  
    7. 결과 영상 저장  
    8. 조건별 프레임당 시간과 최대 처리 속도  


>> [준비] 실습 저장소 받기 (노트북 전용 셀)
<hr>

In [ ]:
#- 실습 저장소 받기: 노트북마다 런타임이 분리되므로 새 세션에서 한 번 실행
import os
if not os.path.exists("npu-course"):
    !git clone -q https://github.com/npu-ondevice-ai/npu-course.git
print("준비 완료:", os.path.exists("npu-course/data"))

# 확인 포인트 ------------------------------------------
# - 준비 완료: True 출력 체크 (이 노트북은 /content 위치에서 실행)

>> [전제] 이 노트북은 두 가지를 이어받습니다. ① `mobilenet_v2.onnx` (없으면 아래 셀이 즉석 변환) ② `mobilenet_v2_quantized_l2.har`와 DFC 전용 환경. NPU 실행모델 변환 실습(n2-2)을 마친 세션에서 이어서 실행하거나, 해당 세션에서 HAR 파일을 Drive에 보관해 두었다가 가져옵니다.

>> [준비 2] ONNX 파일 준비 (노트북 전용 셀)

In [ ]:
#- CPU 백엔드용 ONNX 파일: 없으면 즉석에서 변환
!pip install -q onnx onnxruntime "protobuf<6"
import os
import torch
from torchvision import models

if not os.path.exists("mobilenet_v2.onnx"):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1).eval()
    torch.onnx.export(m, torch.randn(1, 3, 224, 224), "mobilenet_v2.onnx",
                      opset_version=13, input_names=["input"],
                      output_names=["output"], dynamo=False)
print("준비 완료:", os.path.exists("mobilenet_v2.onnx"))

# 확인 포인트 ------------------------------------------
# - 준비 완료: True 출력 체크

>> [준비 3] NPU 에뮬레이터 환경 (노트북 전용 셀, [코드 7-5]부터 필요)

In [ ]:
#- [코드 7-5]의 NPU 에뮬레이터 백엔드가 쓰는 것들: DFC 전용 환경 + 수준 2 HAR
from google.colab import drive
drive.mount('/content/drive')

#- DFC 전용 환경 재구성 (개발 환경 구성 실습과 동일)
WHL = "/content/drive/MyDrive/hailo/hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl"
!pip install -q uv
!uv venv --clear --python 3.10 /content/dfc310
!uv pip install --python /content/dfc310/bin/python "$WHL" 2>&1 | tail -n 1

#- run310: 스크립트를 전용 환경(3.10)으로 실행
import subprocess
def run310(script):
    r = subprocess.run(["/content/dfc310/bin/python", script], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else r.stderr)

#- NPU 실행모델 변환에서 보관한 수준 2 HAR 가져오기
import os
HAR_SRC = "/content/drive/MyDrive/hailo/mobilenet_v2_quantized_l2.har"
if os.path.exists(HAR_SRC):
    !cp "$HAR_SRC" .
print("HAR 준비:", os.path.exists("mobilenet_v2_quantized_l2.har"))

# 확인 포인트 ------------------------------------------
# - HAR 준비: True 체크
# - False면 NPU 실행모델 변환 노트북을 먼저 끝까지 실행해 Drive에 보관 ([보관] 셀 포함)

>> [코드 7-1] 프로젝트 설정 (n2-5_project.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
import os, time, subprocess, numpy as np, pandas as pd, cv2, onnxruntime as ort
from torchvision.models import MobileNet_V2_Weights

#- 클래스 이름과 전처리 상수
CATS = MobileNet_V2_Weights.IMAGENET1K_V1.meta["categories"]   # 클래스 이름 1,000개
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

#- 프로젝트 설정 (경로·샘플링 주기·프레임 상한)
CFG = {
    "video": "npu-course/data/sample_video.mp4",
    "onnx": "mobilenet_v2.onnx",
    "har": "mobilenet_v2_quantized_l2.har",
    "sample_k": 3,        # k프레임마다 한 번 추론
    "max_frames": 90,     # 에뮬레이션 시간을 고려한 추론 프레임 수 상한
    "out_video": "result.mp4",
}

# 확인 포인트 --------------------------------------------------------------------------
# - 출력 없음: CFG의 경로와 k, 상한 값 확인

>> [코드 7-2] 입력 모듈: 프레임 선택 (STEP 2)
<hr>

In [ ]:
#- 입력 모듈: k 프레임마다 골라 224×224 RGB로
#- frames 리스트를 NumPy 배열로 변환하여 반환
def read_frames(path, k, max_frames):
    # 동영상 파일 열기
    cap = cv2.VideoCapture(path)
    frames, idx, i = [], [], 0

    # 선택된 프레임 수가 max_frames에 도달할 때까지 반복
    while len(frames) < max_frames:
        ok, f = cap.read()
        if not ok: break

        # 현재 프레임 번호가 k의 배수이면 선택
        if i % k == 0:
            # 모델 입력 크기에 맞추기 위해 224×224 크기로 축소
            small = cv2.resize(f, (224, 224))
            # BGR → RGB 채널 순서 변환
            frames.append(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
            idx.append(i)
        i += 1

    # 원본 동영상의 FPS(Frame Per Second) 확인
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    # frames 리스트를 NumPy 배열로 변환하여 반환
    return np.stack(frames), idx, i, fps

#- 실행과 선택 결과
# 동영상 프레임 읽기 실행
rgb, idx, n_read, fps = read_frames(
        CFG["video"],           # 입력 동영상 파일 경로
        CFG["sample_k"],        # 몇 프레임마다 하나씩 선택할지
        CFG["max_frames"])      # 최대 몇 장까지 선택할지

print(rgb.shape, f"읽은 {n_read}프레임 중 {len(rgb)}프레임 선택, 원본 {fps:.0f}fps")

# 확인 포인트 -------------------------------------------------------------------------
# - 선택 프레임 수 ≈ (읽은 프레임 수 / k) 체크

>> [코드 7-3] 백엔드별 입력 형식 만들기 (STEP 3)
<hr>

In [ ]:
#- 백엔드별 입력 형식 정의 : cpu는 NCHW·정규화, npu는 NHWC·0~255
def preprocess(rgb, backend):
    # CPU용 전처리 : NCHW, 정규화된 실수
    if backend == "cpu":
        return ((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(0, 3, 1, 2)

    # NPU용 전처리 : NHWC, 0~255 (정규화는 NPU 모델 안에서)
    if backend == "npu":
        return rgb.astype(np.float32)

    # cpu 또는 npu 이외의 값이 들어오면 오류 발생
    raise ValueError(backend)

#- 동일한 RGB 이미지에서 CPU용 / NPU용 입력 데이터를 각각 생성
x_cpu = preprocess(rgb, "cpu")
x_npu = preprocess(rgb, "npu")
print("cpu", x_cpu.shape, f"{x_cpu.min():.2f}~{x_cpu.max():.2f}")
print("npu", x_npu.shape, f"{x_npu.min():.0f}~{x_npu.max():.0f}")

#- NPU용 입력 데이터를 NumPy 파일(.npy)로 저장
np.save("frames_nhwc.npy", x_npu)

# 확인 포인트 -------------------------------------------------------------------------
# - CPU, NPU의 입력 데이터 체크

>> [코드 7-4] CPU 백엔드 (ONNX Runtime, FP32) (STEP 4)
<hr>

In [ ]:
#- ONNX 모델을 CPU에서 실행하기 위한 추론 세션 생성
#- GPU가 아닌 CPU를 사용하여 추론
sess = ort.InferenceSession( CFG["onnx"], providers=["CPUExecutionProvider"])

# 추론 결과 클래스와 프레임별 추론 시간을 저장할 리스트
cpu_pred, cpu_ms = [], []

#- 전처리가 완료된 이미지들을 한 장씩 추론
for x in x_cpu:
    # 추론 시작 시간 기록
    t0 = time.perf_counter()
    # ONNX Runtime으로 모델 추론
    logits = sess.run(None, {"input": x[None]})[0][0]      # 배치 1로 한 장씩
    # 추론에 걸린 시간 계산 및 가장 큰 값을 가진 최종 예측 클래스 번호 선택
    cpu_ms.append((time.perf_counter() - t0) * 1000)
    cpu_pred.append(int(logits.argmax()))

#- 최종 예측 클래스 번호로 선택
cpu_pred = np.array(cpu_pred)

#- 전체 이미지의 평균 CPU 추론 시간 출력
print(f"CPU FP32 추론: 평균 {np.mean(cpu_ms):.1f} ms/프레임")

# 확인 포인트 -------------------------------------------------------------------------
# - 평균 ms/프레임 출력 체크 (절댓값은 머신마다 다름)

>> [코드 7-5] NPU 백엔드 (에뮬레이터, INT8 수준 2) (STEP 5)
<hr>

In [ ]:
%%writefile npu_emu_infer.py

#- 모듈 로딩
import numpy as np
from hailo_sdk_client import ClientRunner, InferenceContext

#- 양자화된 모델 파일
HAR_FILE = "mobilenet_v2_quantized_l2.har"

#- NPU 백엔드: 에뮬레이터에서 INT8 수준 2로 일괄 추론
#  양자화된 모델 로딩
runner = ClientRunner(har=HAR_FILE)

#- NPU 입력 데이터 로딩
x = np.load("frames_nhwc.npy")

#- NPU 에뮬레이터 추론 실행
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as c:
    # 전체 입력 프레임을 한 번에 추론
    out = runner.infer(c, x)

#-추론 결과 후처리
pred = ( np.array(out).reshape(len(x), -1).argmax(1) )

#- NPU 예측 결과 저장
np.save("npu_pred.npy", np.array(out).reshape(len(x), -1).argmax(1))
print("NPU_EMU_OK", len(x))

# 확인 포인트 -------------------------------------------------------------------------
# - Writing npu_emu_infer.py 출력 체크

>> [코드 7-5 실행] NPU 에뮬레이터 추론 실행

In [ ]:
#- 방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("npu_emu_infer.py")

# 확인 포인트 ------------------------------------------
# - NPU_EMU_OK와 프레임 수 출력 체크 (npu_pred.npy 저장)

>> [코드 7-6] 두 경로의 결과 비교 (STEP 6)
<hr>

In [ ]:
#- 두 백엔드의 프레임별 예측 비교
# NPU 에뮬레이터에서 저장한 예측 클래스 번호 불러오기
npu_pred = np.load("npu_pred.npy")

# CPU와 NPU의 예측 일치율 계산
agree = np.mean(cpu_pred == npu_pred) * 100

# 전체 예측 일치율 출력
print(f"CPU와 NPU(에뮬레이터)의 예측 일치율: {agree:.1f}%")

# 5개 프레임에 대해 CPU와 NPU의 실제 예측 클래스 이름 비교
for j in range(5):
    print(f"프레임 {idx[j]:4d}: CPU {CATS[cpu_pred[j]]:20s} / NPU {CATS[npu_pred[j]]}")

# 확인 포인트 -------------------------------------------------------------------------
# - 일치율과 에뮬레이션 검증의 예측 일치율(약 91%) 비교

>> [코드 7-7] 결과 영상 저장 (STEP 7)
<hr>

In [ ]:
#- 원본 영상에 NPU 예측 라벨을 표시하여 새 영상으로 저장

# 원본 동영상 열기 및 해상도 가져오기
cap = cv2.VideoCapture(CFG["video"])
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 출력 동영상 파일 설정
out = cv2.VideoWriter(CFG["out_video"], cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
j, label = 0, ""

#- 추론 프레임에서 라벨 갱신, 나머지는 유지
for i in range(idx[-1] + 1):
    # 원본 영상에서 프레임 한 장 읽기
    ok, f = cap.read()
    if not ok: break

    # 추론한 프레임이 실제 NPU 추론을 수행했던 프레임이면 결과 label을 갱신
    if j < len(idx) and i == idx[j]:
        label = f"NPU: {CATS[npu_pred[j]]}"
        j += 1

    # 현재 프레임에 예측 클래스 문자열 표시
    cv2.putText(f, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    out.write(f)
cap.release(); out.release()
print("저장:", CFG["out_video"])

# 확인 포인트 ------------------------------------------------------------------------
# - 저장: result.mp4 출력 체크

>> [코드 7-8] 조건별 프레임당 시간과 최대 처리 속도 (STEP 8)
<hr>

In [ ]:
#- 파일 상수: 기준 장비 단계별 실측 CSV
E2E_CSV = "npu-course/data/bench/e2e_stages_mobilenet_v2.csv"
#- 기준 장비의 단계별 실측 로딩 (열: setup, run, decode_ms, pre_ms, infer_ms, post_ms)
e2e = pd.read_csv(E2E_CSV)
#- setup별 평균 지연시간 계산
st = e2e.groupby("setup")[["decode_ms", "pre_ms", "infer_ms", "post_ms"]].mean()

#- 구성별 프레임당 시간: 디코드는 매 프레임, 나머지는 추론 프레임(1/k)에만
def per_frame(setup, k):
    s = st.loc[setup]
    return s["decode_ms"] + (s["pre_ms"] + s["infer_ms"] + s["post_ms"]) / k

#- 세 구성 비교 (영상 30fps 기준의 결과 갱신 횟수)
for name, setup, k in [("A", "cpu", 1), ("B", "npu", 1), ("C", "npu", 3)]:
    # 해당 구성의 평균 프레임당 처리 시간 계산
    t = per_frame(setup, k)
    print(f"{name}: 프레임당 {t:.2f} ms, 순차 처리 기준 {1000 / t:.0f} fps, ", end="")
    print(f"결과 갱신 {30 / k:.0f}회/초")

# 확인 포인트 -------------------------------------------------------------------------
# - 세 구성의 프레임당 ms와 최대 fps 출력 체크